In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
import warnings
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

warnings.filterwarnings("ignore")

In [ ]:
# 1. Carga del dataset con las probabilidades limpias del Bloque 1
df = pd.read_excel("Data/dataset_con_probabilidades_2.xlsx")
df

In [ ]:


# 2. Selección de variables predictoras y objetivo
variables_predictoras = [
    "Edad", "Ingresos", "Monto_Inicial", "Scoring_Crediticio",
    "Ratio_Interes", "Duracion", "Ratio_Deuda_Ingresos", 
    "Estudios", "Estado_Civil_Casado", "prob_impago"
]

X = df[variables_predictoras]
y = df["Prima"]

# 3. División en conjunto de entrenamiento y prueba (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    train_size=0.8, 
    random_state=42, 
    shuffle=True
)

print(f"Observaciones de entrenamiento: {len(X_train)}")
print(f"Observaciones de validación/test: {len(X_test)}")

In [ ]:
resultados_grados = []
datos_por_grado = {}

for grado in [1, 2, 3]:
    print("============================================================")
    print(f" Evaluando Polinomio Base (Sin Regularización) | Grado: {grado}")
    print("============================================================")
    
    # 1. Transformación Polinómica (Ajustamos en Train, transformamos en Test)
    poly = PolynomialFeatures(degree=grado, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    nombres_poly = poly.get_feature_names_out(X_train.columns)
    
    # 2. Estandarización Secuencial
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_poly), columns=nombres_poly, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_poly), columns=nombres_poly, index=X_test.index)
    
    # Guardamos los datos en memoria para reutilizarlos directamente en la Fase 2
    datos_por_grado[grado] = {
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "poly": poly,
        "scaler": scaler
    }
    
    # 3. Entrenamiento sin regularización (Mínimos Cuadrados Ordinarios)
    modelo_base = LinearRegression(n_jobs=-1)
    modelo_base.fit(X_train_scaled, y_train)
    
    # 4. Predicciones en Train y Test
    pred_train = modelo_base.predict(X_train_scaled)
    pred_test = modelo_base.predict(X_test_scaled)
    
    # 5. Cálculo de métricas y diferencia entre Train y Test
    rmse_train = root_mean_squared_error(y_train, pred_train)
    rmse_test = root_mean_squared_error(y_test, pred_test)
    dif_rmse = rmse_test - rmse_train
    
    mae_train = mean_absolute_error(y_train, pred_train)
    mae_test = mean_absolute_error(y_test, pred_test)
    
    mape_train = mean_absolute_percentage_error(y_train, pred_train)
    mape_test = mean_absolute_percentage_error(y_test, pred_test)
    
    print(f"-> Variables polinómicas generadas: {X_train_scaled.shape[1]}")
    print(f"-> RMSE Train: {rmse_train:.4f} | RMSE Test: {rmse_test:.4f} | Diferencia (Test - Train): {dif_rmse:.4f}")
    print(f"-> MAE Train:  {mae_train:.4f} | MAE Test:  {mae_test:.4f}")
    print(f"-> MAPE Train: {mape_train:.2%} | MAPE Test: {mape_test:.2%}\n")
    
    resultados_grados.append({
        "Grado": grado,
        "Num_Variables": X_train_scaled.shape[1],
        "RMSE_Train": round(rmse_train, 4),
        "RMSE_Test": round(rmse_test, 4),
        "Diferencia_RMSE": round(dif_rmse, 4),
        "MAE_Train": round(mae_train, 4),
        "MAE_Test": round(mae_test, 4),
        "MAPE_Train (%)": round(mape_train * 100, 2),
        "MAPE_Test (%)": round(mape_test * 100, 2)
    })

# Mostramos la tabla comparativa para que tomes la decisión
df_comparativa_grados = pd.DataFrame(resultados_grados)
print("--- TABLA RESUMEN FASE 1: ELECCIÓN DE GRADO ---")
display(df_comparativa_grados)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_comparativa_grados["Grado"], df_comparativa_grados["RMSE_Train"], "r-o", linewidth=2, label="RMSE Train")
plt.plot(df_comparativa_grados["Grado"], df_comparativa_grados["RMSE_Test"], "b-s", linewidth=2, label="RMSE Test")
plt.title("Evolución del RMSE en Train y Test según el Grado Polinómico")
plt.xlabel("Grado del Polinomio")
plt.ylabel("RMSE")
plt.xticks([1, 2, 3])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
resultados_grados = []
datos_por_grado = {}

for grado in [1, 2, 3]:
    print("============================================================")
    print(f" Evaluando Polinomio Base (Sin Regularización) | Grado: {grado}")
    print("============================================================")
    
    # 1. Transformación Polinómica (Ajustamos en Train, transformamos en Test)
    poly = PolynomialFeatures(degree=grado, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    nombres_poly = poly.get_feature_names_out(X_train.columns)
    
    # 2. Estandarización Secuencial
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_poly), columns=nombres_poly, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_poly), columns=nombres_poly, index=X_test.index)
    
    # Guardamos los datos en memoria para reutilizarlos directamente en la Fase 2
    datos_por_grado[grado] = {
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "poly": poly,
        "scaler": scaler
    }
    
    # 3. Entrenamiento sin regularización (Mínimos Cuadrados Ordinarios)
    modelo_base = LinearRegression(n_jobs=-1)
    modelo_base.fit(X_train_scaled, y_train)
    
    # 4. Predicciones en Train y Test
    pred_train = modelo_base.predict(X_train_scaled)
    pred_test = modelo_base.predict(X_test_scaled)
    
    # 5. Cálculo de métricas y diferencia entre Train y Test
    rmse_train = root_mean_squared_error(y_train, pred_train)
    rmse_test = root_mean_squared_error(y_test, pred_test)
    dif_rmse = rmse_test - rmse_train
    
    mae_train = mean_absolute_error(y_train, pred_train)
    mae_test = mean_absolute_error(y_test, pred_test)
    
    mape_train = mean_absolute_percentage_error(y_train, pred_train)
    mape_test = mean_absolute_percentage_error(y_test, pred_test)
    
    print(f"-> Variables polinómicas generadas: {X_train_scaled.shape[1]}")
    print(f"-> RMSE Train: {rmse_train:.4f} | RMSE Test: {rmse_test:.4f} | Diferencia (Test - Train): {dif_rmse:.4f}")
    print(f"-> MAE Train:  {mae_train:.4f} | MAE Test:  {mae_test:.4f}")
    print(f"-> MAPE Train: {mape_train:.2%} | MAPE Test: {mape_test:.2%}\n")
    
    resultados_grados.append({
        "Grado": grado,
        "Num_Variables": X_train_scaled.shape[1],
        "RMSE_Train": round(rmse_train, 4),
        "RMSE_Test": round(rmse_test, 4),
        "Diferencia_RMSE": round(dif_rmse, 4),
        "MAE_Train": round(mae_train, 4),
        "MAE_Test": round(mae_test, 4),
        "MAPE_Train (%)": round(mape_train * 100, 2),
        "MAPE_Test (%)": round(mape_test * 100, 2)
    })

# Mostramos la tabla comparativa para que tomes la decisión
df_comparativa_grados = pd.DataFrame(resultados_grados)
print("--- TABLA RESUMEN FASE 1: ELECCIÓN DE GRADO ---")
display(df_comparativa_grados)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_comparativa_grados["Grado"], df_comparativa_grados["RMSE_Train"], "r-o", linewidth=2, label="RMSE Train")
plt.plot(df_comparativa_grados["Grado"], df_comparativa_grados["RMSE_Test"], "b-s", linewidth=2, label="RMSE Test")
plt.title("Evolución del RMSE en Train y Test según el Grado Polinómico")
plt.xlabel("Grado del Polinomio")
plt.ylabel("RMSE")
plt.xticks([1, 2, 3])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ==============================================================================
# FASE 2 OPTIMIZADA: Aplicación de Regularización al Grado Ganador
# ==============================================================================
grado_ganador = 3

# Recuperamos los datos ya transformados de ese grado
X_train_ganador = datos_por_grado[grado_ganador]["X_train_scaled"]
X_test_ganador = datos_por_grado[grado_ganador]["X_test_scaled"]

# Diccionario de modelos (con los parámetros ajustados para cálculo de alphas automáticos)
modelos_regularizacion = {
    "Lasso (L1) Rapido": LassoCV(
        n_alphas=100, 
        cv=5,
        max_iter=5000,
        tol=0.01,
        n_jobs=-1,
        random_state=42
    )
}

resultados_regularizacion = []

# 1. Guardamos primero la fila del modelo Sin Regularización como referencia
fila_base = df_comparativa_grados[df_comparativa_grados["Grado"] == grado_ganador].iloc[0]
resultados_regularizacion.append({
    "Modelo": "Sin Regularización (Base)",
    "Grado": grado_ganador,
    "Mejor_Parametro": "N/A",
    "Vars_Activas": int(fila_base["Num_Variables"]),
    "Vars_Eliminadas": 0,
    "RMSE_Train": fila_base["RMSE_Train"],
    "RMSE_Test": fila_base["RMSE_Test"],
    "Diferencia_RMSE": fila_base["Diferencia_RMSE"],
    "MAE_Train": fila_base["MAE_Train"],
    "MAE_Test": fila_base["MAE_Test"],
    "MAPE_Train (%)": fila_base["MAPE_Train (%)"],
    "MAPE_Test (%)": fila_base["MAPE_Test (%)"]
})

# 2. Entrenamos y evaluamos iterando sobre el diccionario de modelos
for nombre_reg, modelo in modelos_regularizacion.items():
    print("============================================================")
    print(f" Evaluando Regularización: {nombre_reg} | Grado: {grado_ganador}")
    print("============================================================")
    
    # Ajuste del modelo
    modelo.fit(X_train_ganador, y_train)
    
    # Análisis de variables eliminadas (coeficiente = 0)
    vars_totales = X_train_ganador.shape[1]
    vars_cero = int(np.sum(modelo.coef_ == 0))
    vars_activas = vars_totales - vars_cero
    print(f"-> Variables eliminadas por Lasso: {vars_cero} de {vars_totales}")
    print(f"-> Mejor parámetro alpha encontrado: {modelo.alpha_:.4f}")
    
    # Predicciones
    pred_train = modelo.predict(X_train_ganador)
    pred_test = modelo.predict(X_test_ganador)
    
    # ---> CÁLCULO DE RESIDUOS (Necesario para los tests estadísticos posteriores) <---
    residuos_train = y_train - pred_train
    
    # Cálculo de métricas
    rmse_train = root_mean_squared_error(y_train, pred_train)
    rmse_test = root_mean_squared_error(y_test, pred_test)
    dif_rmse = rmse_test - rmse_train
    
    mae_train = mean_absolute_error(y_train, pred_train)
    mae_test = mean_absolute_error(y_test, pred_test)
    
    mape_train = mean_absolute_percentage_error(y_train, pred_train) * 100
    mape_test = mean_absolute_percentage_error(y_test, pred_test) * 100
    
    resultados_regularizacion.append({
        "Modelo": nombre_reg,
        "Grado": grado_ganador,
        "Mejor_Parametro": round(modelo.alpha_, 4),
        "Vars_Activas": vars_activas,
        "Vars_Eliminadas": vars_cero,
        "RMSE_Train": round(rmse_train, 4),
        "RMSE_Test": round(rmse_test, 4),
        "Diferencia_RMSE": round(dif_rmse, 4),
        "MAE_Train": round(mae_train, 4),
        "MAE_Test": round(mae_test, 4),
        "MAPE_Train (%)": round(mape_train, 2),
        "MAPE_Test (%)": round(mape_test, 2)
    })

# 3. Tabla comparativa final
dataframe_resultados_regularizacion = pd.DataFrame(resultados_regularizacion)
display(dataframe_resultados_regularizacion)